# 🧾 Day 3 SQL and PySpark Challenge

## 📌 Task

Write a query to:

### 🎯 Calculate total revenue
Use:
- `SUM(totalPrice)` for each franchise

---

## 📊 Include the following columns:
- franchiseID  
- franchise name  
- city  
- total_revenue  

---

## 🔍 Filter condition:
Only include transactions where:
- `paymentMethod = 'mastercard'`

---

## 📈 Sorting:
- Order by **highest revenue first**

---

## 🏁 Limit:
- Show **Top 5 franchises**

In [0]:
%sql

select * from samples.bakehouse.sales_transactions limit 10;
select * from samples.bakehouse.sales_franchises limit 10;

transactionID,customerID,franchiseID,dateTime,product,quantity,unitPrice,totalPrice,paymentMethod,cardNumber
1002961,2000253,3000047,2024-05-14T12:17:01.495Z,Golden Gate Ginger,8,3,24,amex,378154478982993
1003007,2000226,3000047,2024-05-10T23:10:10.239Z,Austin Almond Biscotti,36,3,108,mastercard,2244626981238094
1003017,2000108,3000047,2024-05-16T16:34:10.613Z,Austin Almond Biscotti,40,3,120,mastercard,2490570234487424
1003068,2000173,3000047,2024-05-02T04:31:51.612Z,Pearly Pies,28,3,84,amex,343808569426192
1003103,2000075,3000047,2024-05-04T23:44:26.902Z,Pearly Pies,28,3,84,visa,4377080942201798
1003147,2000295,3000047,2024-05-15T16:17:06.259Z,Austin Almond Biscotti,32,3,96,amex,371093774812677
1003196,2000237,3000047,2024-05-07T11:13:22.469Z,Tokyo Tidbits,40,3,120,mastercard,5538807345848392
1003329,2000272,3000047,2024-05-06T03:32:16.017Z,Outback Oatmeal,28,3,84,visa,4872480716880043
1001264,2000209,3000047,2024-05-16T17:32:28.547Z,Pearly Pies,28,3,84,mastercard,5287105980593305
1001287,2000120,3000047,2024-05-15T08:41:28.406Z,Austin Almond Biscotti,40,3,120,amex,376211012259783


In [0]:
%sql

WITH cte AS (
SELECT
f.franchiseID,
f.name,
f.city,
t.totalPrice
FROM samples.bakehouse.sales_transactions t
JOIN samples.bakehouse.sales_franchises f
ON t.franchiseID = f.franchiseID where t.paymentMethod = 'mastercard'
)

SELECT
franchiseID,
name,
city,
SUM(totalPrice) as total_revenue
FROM cte
GROUP BY franchiseID, name, city
ORDER BY total_revenue DESC limit 5

franchiseID,name,city,totalPrice
3000046,Baked Bliss,Rome,2052
3000047,Sweet Sinsations,Stockholm,1260
3000000,The Crumbly Nook,Sydney,1014
3000002,Golden Crumbs,San Francisco,966
3000021,Butter Bliss,Honolulu,882


In [0]:
df_transactions = spark.table("samples.bakehouse.sales_transactions")
df_franchises = spark.table("samples.bakehouse.sales_franchises")

In [0]:
from pyspark.sql import functions as F

df = df_transactions.join(df_franchises, "franchiseID", "inner").filter(df_transactions.paymentMethod == "mastercard").select(df_franchises.franchiseID,df_franchises.name,df_franchises.city,"totalPrice")
#display(df)

In [0]:

df = df.groupBy("franchiseID", "name", "city").agg(F.sum("totalPrice").alias("total_revenue")).orderBy(F.col("total_revenue").desc())

df.show(5)

+-----------+----------------+-------------+-------------+
|franchiseID|            name|         city|total_revenue|
+-----------+----------------+-------------+-------------+
|    3000046|     Baked Bliss|         Rome|         2052|
|    3000047|Sweet Sinsations|    Stockholm|         1260|
|    3000000|The Crumbly Nook|       Sydney|         1014|
|    3000002|   Golden Crumbs|San Francisco|          966|
|    3000021|    Butter Bliss|     Honolulu|          882|
+-----------+----------------+-------------+-------------+
only showing top 5 rows


In [0]:
from pyspark.sql import functions as F

df = df_transactions \
    .join(df_franchises, "franchiseID", "inner") \
    .filter(F.col("paymentMethod") == "mastercard") \
    .groupBy("franchiseID", "name", "city") \
    .agg(F.sum("totalPrice").alias("total_revenue")) \
    .orderBy(F.col("total_revenue").desc())

df.show(5)

+-----------+----------------+-------------+-------------+
|franchiseID|            name|         city|total_revenue|
+-----------+----------------+-------------+-------------+
|    3000046|     Baked Bliss|         Rome|         2052|
|    3000047|Sweet Sinsations|    Stockholm|         1260|
|    3000000|The Crumbly Nook|       Sydney|         1014|
|    3000002|   Golden Crumbs|San Francisco|          966|
|    3000021|    Butter Bliss|     Honolulu|          882|
+-----------+----------------+-------------+-------------+
only showing top 5 rows


# 🐍 Day 3 Python Coding Challenge — Strings + Aggregation

## 📌 Problem: Sales Receipt Analyzer

You are given a list of transaction strings.  
Each string contains transaction details in the following format:

```
customerID,product,quantity,unit_price
```

---

## 📊 Example Input

```python
transactions = [
    "C001,Apple,4,10",
    "C002,Banana,2,5",
    "C001,Mango,3,20",
    "C003,Apple,1,10"
]
```

---

## 🎯 Task

Write a Python program to:

### 1. Parse each transaction
Split each string into:
- customerID  
- product  
- quantity  
- unit_price  

### 2. Calculate transaction value
```
total_price = quantity * unit_price
```

### 3. Aggregate data
Compute **total spending per customer**

### 4. Final Output
Print the **top 2 highest spending customers**

---

## 📌 Expected Output

```
C001: 100
C002: 10
```

---


In [0]:
transactions = [
    "C001,Apple,4,10",
    "C002,Banana,2,5",
    "C001,Mango,3,20",
    "C003,Apple,1,10"
]

rows = [row.split(",") for row in transactions]
print(rows)
  


[['C001', 'Apple', '4', '10'], ['C002', 'Banana', '2', '5'], ['C001', 'Mango', '3', '20'], ['C003', 'Apple', '1', '10']]


In [0]:
from pyspark.sql import functions as F

df1 = spark.createDataFrame(rows, ["customerID", "product", "quantity", "unit_price"]) \
    .withColumn("quantity", F.col("quantity").cast("int")) \
    .withColumn("unit_price", F.col("unit_price").cast("int"))
display(df1)

customerID,product,quantity,unit_price
C001,Apple,4,10
C002,Banana,2,5
C001,Mango,3,20
C003,Apple,1,10


In [0]:
df1 = df1.withColumn("total_price", F.col("quantity") * F.col("unit_price"))
display(df1)

customerID,product,quantity,unit_price,total_price
C001,Apple,4,10,40
C002,Banana,2,5,10
C001,Mango,3,20,60
C003,Apple,1,10,10


In [0]:
df1 = df1.groupBy("customerID").agg(F.sum("total_price").alias("total_price_per_customer")).orderBy(F.col("total_price_per_customer").desc())
display(df1)

customerID,total_price_per_customer
C001,100
C002,10
C003,10


# 🐍 Day 3 Python Challenge — Inheritance (OOP)

## 📌 Problem: Employee Salary System

You are building a simple employee management system using **Object-Oriented Programming (OOP)**.

---

## 🏗️ Base Class: Employee

Create a class called `Employee` with the following:

### Attributes:
- `name`
- `employee_id`
- `base_salary`

### Methods:
- `get_details()` → returns employee details
- `calculate_salary()` → returns base salary

---

## 👨‍💻 Derived Class: Manager

Create a class `Manager` that **inherits from Employee**.

### Additional Attribute:
- `bonus`

### Override Method:
- `calculate_salary()` → returns:
```
base_salary + bonus
```

---

## 🎯 Task

1. Create an object of `Manager`
2. Set values for:
   - name
   - employee_id
   - base_salary
   - bonus
3. Print:
   - Employee details
   - Final salary

---

## 📌 Expected Output Example

```
Name: John
Employee ID: E101
Base Salary: 50000
Total Salary: 60000
```

---



In [0]:
class Employee:
    def __init__(self, name, employee_id, base_salary):
        self.name = name
        self.employee_id = employee_id
        self.base_salary = base_salary
   
    def get_details(self):
        return f"Employee: {self.name}, ID: {self.employee_id}"
    
    def calculate_salary(self):
        return self.base_salary

class manager(Employee):
    def __init__(self, name, employee_id, base_salary, bonus):
        super().__init__(name, employee_id, base_salary)
        self.bonus = bonus
    
    def calculate_salary(self):
        return self.base_salary + self.bonus
employee1 = Employee("John", 1, 50000)
employee2 = manager("Jane", 2, 60000, 10000)
print(employee1.get_details())
print(employee2.get_details())
print(employee1.calculate_salary())
print(employee2.calculate_salary())



Employee: John, ID: 1
Employee: Jane, ID: 2
50000
70000
